In [0]:
# =================================================================
# NOTEBOOK: 01_TRAINING_MODELS_ENERGY
# Objetivo: Entrenar modelos de Consumo, CO2 y Clasificación
# =================================================================

from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, Imputer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

# 1. CARGA DE DATOS DESDE CAPA GOLD
# -----------------------------------------------------------------
print("📥 Cargando datos desde la capa Gold...")
train_df = spark.table("workspace.default.gold_train_eff_ener")
test_df = spark.table("workspace.default.gold_test_eff_ene")

# 2. LIMPIEZA Y VALIDACIÓN
# -----------------------------------------------------------------
label_cols = ["consumo_energia", "emision_co2", "clase_consumo"]
train_clean = train_df.dropna(subset=label_cols)
test_clean = test_df.dropna(subset=label_cols)

print(f"✅ Datos Train: {train_clean.count()} registros.")
print(f"✅ Datos Test: {test_clean.count()} registros.")

# 3. MANEJO DE VALORES NULOS EN FEATURES
# -----------------------------------------------------------------
features_cols = ["superficie", "anio_construccion", "antiguedad", "latitud", "longitud"]

# Imputar valores faltantes con la mediana
imputer = Imputer(
    inputCols=features_cols,
    outputCols=[f"{col}_imputed" for col in features_cols],
    strategy="median"
)

imputer_model = imputer.fit(train_clean)
train_imputed = imputer_model.transform(train_clean)
test_imputed = imputer_model.transform(test_clean)

imputed_cols = [f"{col}_imputed" for col in features_cols]
print(f"✅ Imputación completada para {len(features_cols)} columnas.")

# 4. PREPARACIÓN DE FEATURES (VECTORIZACIÓN Y ESCALADO)
# -----------------------------------------------------------------
assembler = VectorAssembler(
    inputCols=imputed_cols, 
    outputCol="features_unscaled",
    handleInvalid="skip"
)

train_vector = assembler.transform(train_imputed)
test_vector = assembler.transform(test_imputed)

scaler = StandardScaler(
    inputCol="features_unscaled", 
    outputCol="features", 
    withStd=True, 
    withMean=False
)

scaler_model = scaler.fit(train_vector)
train_final = scaler_model.transform(train_vector)
test_final = scaler_model.transform(test_vector)

print(f"✅ Features preparados - Train: {train_final.count()}, Test: {test_final.count()} registros.")

# 5. OBJETIVO 1: PREDICCIÓN DE CONSUMO ENERGÉTICO (Regresión)
# -----------------------------------------------------------------
print("\n" + "="*80)
print("🚀 OBJETIVO 1: PREDICCIÓN DE CONSUMO ENERGÉTICO")
print("="*80)

rf_consumo = RandomForestRegressor(
    featuresCol="features", 
    labelCol="consumo_energia", 
    numTrees=100,
    maxDepth=10,
    seed=42
)
model_consumo = rf_consumo.fit(train_final)
pred_consumo = model_consumo.transform(test_final)

evaluator_rmse = RegressionEvaluator(labelCol="consumo_energia", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="consumo_energia", predictionCol="prediction", metricName="r2")

rmse_consumo = evaluator_rmse.evaluate(pred_consumo)
r2_consumo = evaluator_r2.evaluate(pred_consumo)

print(f"\n✅ Resultados Consumo Energético:")
print(f"   RMSE: {rmse_consumo:.2f}")
print(f"   R²: {r2_consumo:.4f}")

print("\nMuestra de predicciones:")
pred_consumo.select("numero_certificado", "consumo_energia", "prediction", "superficie", "antiguedad").show(10)

# 6. OBJETIVO 2: PREDICCIÓN DE EMISIONES CO2 (Regresión)
# -----------------------------------------------------------------
print("\n" + "="*80)
print("🚀 OBJETIVO 2: PREDICCIÓN DE EMISIONES CO2")
print("="*80)

rf_co2 = RandomForestRegressor(
    featuresCol="features", 
    labelCol="emision_co2", 
    numTrees=100,
    maxDepth=10,
    seed=42
)
model_co2 = rf_co2.fit(train_final)
pred_co2 = model_co2.transform(test_final)

rmse_co2 = RegressionEvaluator(labelCol="emision_co2", predictionCol="prediction", metricName="rmse").evaluate(pred_co2)
r2_co2 = RegressionEvaluator(labelCol="emision_co2", predictionCol="prediction", metricName="r2").evaluate(pred_co2)

print(f"\n✅ Resultados Emisiones CO2:")
print(f"   RMSE: {rmse_co2:.2f}")
print(f"   R²: {r2_co2:.4f}")

print("\nMuestra de predicciones:")
pred_co2.select("numero_certificado", "emision_co2", "prediction", "superficie", "antiguedad").show(10)

# 7. OBJETIVO 3: CLASIFICACIÓN DE EFICIENCIA ENERGÉTICA
# -----------------------------------------------------------------
print("\n" + "="*80)
print("🚀 OBJETIVO 3: CLASIFICACIÓN DE EFICIENCIA ENERGÉTICA")
print("="*80)

indexer = StringIndexer(inputCol="clase_consumo", outputCol="label_clase")
indexer_model = indexer.fit(train_final)
train_classed = indexer_model.transform(train_final)
test_classed = indexer_model.transform(test_final)

rf_clase = RandomForestClassifier(
    featuresCol="features", 
    labelCol="label_clase", 
    numTrees=100,
    maxDepth=10,
    seed=42
)
model_clase = rf_clase.fit(train_classed)
pred_clase = model_clase.transform(test_classed)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label_clase", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_clase", predictionCol="prediction", metricName="f1")

accuracy = evaluator_acc.evaluate(pred_clase)
f1_score = evaluator_f1.evaluate(pred_clase)

print(f"\n✅ Resultados Clasificación:")
print(f"   Accuracy: {accuracy:.4f}")
print(f"   F1-Score: {f1_score:.4f}")

print("\nMuestra de predicciones (clase):")
pred_clase.select("numero_certificado", "clase_consumo", "label_clase", "prediction").show(10)

# 8. OBJETIVO 4: IDENTIFICACIÓN DE FACTORES CLAVE
# -----------------------------------------------------------------
print("\n" + "="*80)
print("📊 OBJETIVO 4: FACTORES CLAVE (Feature Importance)")
print("="*80)

importances_consumo = model_consumo.featureImportances
importances_co2 = model_co2.featureImportances

print("\nImportancia para Consumo Energético:")
for i, col_name in enumerate(features_cols):
    print(f"  {col_name}: {importances_consumo[i]:.4f}")

print("\nImportancia para Emisiones CO2:")
for i, col_name in enumerate(features_cols):
    print(f"  {col_name}: {importances_co2[i]:.4f}")

print("\n✅ ENTRENAMIENTO COMPLETADO - Modelos listos para usar!")